In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import bacco
from matplotlib.colors import LogNorm
from scipy.ndimage import gaussian_filter

In [ ]:
## Load the Zooms
sigma8 = 0.8159 #CHECK ME
ns     = 0.9667 #CHECK ME
tau    = 0.0965 #CHECK ME

#name_list = ['LH_{:d}'.format(i) for i in range(30)] + ['fiducial'] + ['bf_sim']
name_list = ['bf_sim']

snap = 264

name_list = ['fiducial']

zoom = {}
for i in range(len(name_list)):
    base = "/cosmos_storage/simulations/TNG_Family/MN5_resims/"+name_list[i]+"/hydro_output/"
    zoom[name_list[i]] = bacco.Simulation(basedir=base, halo_file="groups_{:03d}/fof_subhalo_tab_{:03d}".format(snap,snap), sim_format='TNG500', fixedPk=True, use_orphans=False,\
                            tau=tau, ns=ns, sigma8=sigma8, dm_file="snapdir_{:03d}/snapshot_{:03d}".format(snap,snap), use_ids=True, numpart=4320)



In [ ]:
zoom['fiducial'].gas

In [ ]:
f_b = zoom['fiducial'].Cosmology.pars['omega_baryon'] / zoom['fiducial'].Cosmology.pars['omega_matter']

In [ ]:
fig, ax = plt.subplots(dpi=200, figsize=(5.5,5))

ax.set_ylabel('Number of cells')
ax.set_xlabel('$M_{gas}$ [$M_\odot$]')
ax.set_xscale('log')
ax.hist(zoom['fiducial'].gas['mass']*1e10, bins=np.logspace(6,12.5,30), log=True)
ax.axvline(zoom['fiducial'].header['ParticleMass'] * 1e10 * f_b, color='k', ls='--', label='$m_{\mathrm{gas}}$')
ax.axvline(zoom['fiducial'].header['ParticleMass'] * (4320/128)**3 * 1e10 * f_b, color='C3', ls='--', label='$m_{\mathrm{gas}} \\times (4320/128)^3$')

ax.legend()

In [ ]:
center = zoom['fiducial'].fof['halo_pos'][0,:]

In [ ]:
gas_pos = zoom['fiducial'].gas['pos'] - center
gas_mass = zoom['fiducial'].gas['mass']*1e10
mask_gas = ( np.abs(gas_pos[:,0])<20 ) & ( np.abs(gas_pos[:,1]) < 20) & ( np.abs(gas_pos[:,2]) < 20)

In [ ]:
dm_pos = zoom['fiducial'].dm['pos'] - center
dm_mass = 1e10 * np.ones(dm_pos.shape[0]) * zoom['fiducial'].header['ParticleMass']
mask = ( np.abs(dm_pos[:,0])<20 ) & ( np.abs(dm_pos[:,1]) < 20) & ( np.abs(dm_pos[:,2]) < 20) 

xy_range = [[-20, 20], [-20, 20]]

cell = 40 / 1000

hist_dm = np.histogram2d(dm_pos[mask][:,0], dm_pos[mask][:,1], weights=dm_mass[mask], bins=1000, range=xy_range)

In [ ]:
fig, ax = plt.subplots(dpi=200, figsize=(6, 5))

# --- per-direction distance cut, combined with your existing mask ---
d = np.abs(gas_pos)                                   # |x|, |y|, (|z|) from center=0
shell = np.all((d >= 10) & (d <= 20), axis=1)         # in [10,20] along EVERY axis
mask = shell

# --- DM background (unchanged) ---
h_dm = hist_dm[0].T / cell**3
box_size = 40
extent = [-box_size/2, box_size/2, -box_size/2, box_size/2]

im_dm = ax.imshow(h_dm, cmap='inferno',
                  norm=LogNorm(vmin=np.median(dm_mass), vmax=np.max(h_dm)),
                  origin='lower', extent=extent)

ax.set_xlabel("x [Mpc/h]")
ax.set_ylabel("y [Mpc/h]")
ax.set_title(r"DM Density Projection ($\Delta z = 250$ [Mpc/$h$])")

cbar = fig.colorbar(im_dm, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label(r'High-Res DM Density [$M_\odot h^{-1} / ($Mpc$/h)^3$]')

# --- gas scatter, only the selected shell ---
ax.scatter(gas_pos[mask, 0], gas_pos[mask, 1],
           s=gas_mass[mask] / ( 300 * dm_mass[0] * f_b), color='C3')

plt.tight_layout()
plt.show()